# OOD and Query-Image Scoring Experiments

Compare out-of-distribution scoring methods using public texture datasets.
One dataset serves as the in-distribution reference; the other as OOD queries.

> **Warning:** Results are diagnostic only. FabricNET and LeaData are public
> texture/material datasets, not luxury authentication datasets. OOD separation
> here measures cross-dataset domain difference, not authentic-vs-counterfeit
> detection.

## Run Experiments

From `auth-platform/`:

```bash
# FabricNET as ID, LeaData as OOD
make ood-experiment-fabricnet-vs-leadata

# LeaData as ID, FabricNET as OOD
make ood-experiment-leadata-vs-fabricnet

# Compare all
python3 scripts/compare_ood_experiments.py --latest
```

In [ ]:
import json
from pathlib import Path

%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

OOD_DIR = Path("../experiments/outputs/ood")

# Find available summaries
summaries = sorted(OOD_DIR.glob("*__ood_summary.json")) if OOD_DIR.exists() else []
print(f"Found {len(summaries)} OOD experiment summaries")
for s in summaries:
    print(f"  {s.name}")

In [ ]:
# Load first available summary
if not summaries:
    print("No OOD summaries found. Run experiments first.")
    data = None
else:
    data = json.loads(summaries[0].read_text())
    print(f"In-distribution: {data['in_dataset']}")
    print(f"OOD:             {data['out_dataset']}")
    print(f"Model:           {data['model']}")
    print(f"ID queries:      {data['id_count']}")
    print(f"OOD queries:     {data['ood_count']}")
    print(f"Best method:     {data['best_method']}")
    print()

    # Method comparison table
    methods = ["max_similarity", "mean_topk", "similarity_gap", "knn_density", "mahalanobis", "lof"]
    header = f"{'Method':<18} {'AUROC':>7} {'ID mean':>8} {'OOD mean':>9} {'Sep':>7} {'Overlap':>8}"
    print(header)
    print("-" * len(header))
    for m in methods:
        s = data['method_summaries'].get(m, {})
        if s:
            print(f"{m:<18} {s['auroc']:>7.4f} {s['id_mean']:>8.4f} {s['ood_mean']:>9.4f} "
                  f"{s['separation']:>7.4f} {s['overlap']:>8.4f}")

In [ ]:
# Load per-query scores for distribution plots
import csv

scores_files = sorted(OOD_DIR.glob("*__ood_scores.csv")) if OOD_DIR.exists() else []
if scores_files:
    with open(scores_files[0]) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    id_rows = [r for r in rows if r["type"] == "ID"]
    ood_rows = [r for r in rows if r["type"] == "OOD"]
    print(f"Loaded {len(id_rows)} ID and {len(ood_rows)} OOD score rows")
else:
    id_rows, ood_rows = [], []
    print("No score files found.")

In [ ]:
# Distribution plot: max_similarity OOD score
if id_rows and ood_rows:
    id_max = [float(r["max_similarity_ood_score"]) for r in id_rows]
    ood_max = [float(r["max_similarity_ood_score"]) for r in ood_rows]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(id_max, bins=30, alpha=0.6, label="In-distribution", color="steelblue")
    ax.hist(ood_max, bins=30, alpha=0.6, label="OOD", color="tomato")
    ax.set_xlabel("OOD Score (1 - max similarity)")
    ax.set_ylabel("Count")
    ax.set_title("Max Similarity OOD Score Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
# Distribution plot: mean_topk OOD score
if id_rows and ood_rows:
    id_topk = [float(r["mean_topk_ood_score"]) for r in id_rows]
    ood_topk = [float(r["mean_topk_ood_score"]) for r in ood_rows]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(id_topk, bins=30, alpha=0.6, label="In-distribution", color="steelblue")
    ax.hist(ood_topk, bins=30, alpha=0.6, label="OOD", color="tomato")
    ax.set_xlabel("OOD Score (1 - mean top-K similarity)")
    ax.set_ylabel("Count")
    ax.set_title("Mean Top-K OOD Score Distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# AUROC comparison bar chart
if data:
    methods_plot = [m for m in methods if m in data['method_summaries']]
    aurocs = [data['method_summaries'][m]['auroc'] for m in methods_plot]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(methods_plot, aurocs, color="darkorange")
    ax.set_xlabel("AUROC")
    ax.set_title("AUROC by OOD Scoring Method")
    ax.set_xlim(0, 1.05)
    for bar, val in zip(bars, aurocs):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

## Notes / Conclusions

- OOD scoring methods are compared on public texture datasets.
- The ID/OOD distinction is between dataset domains (fabric vs leather texture),
  not between authentic and counterfeit luxury items.
- Current query-image UI uses simple similarity-based interpretation bands.
  Dedicated OOD scoring is experimental and not yet promoted to the UI.
- Real OOD detection for luxury authentication requires microscope/gold-set data.

**These results are diagnostic only.** They do not prove authentication accuracy.